# 01 Data inventory and integrity
What is on disk, whether the scored tract list agrees across every source, what the submission template looks like, and what each region is.


In [1]:
from common import *
import pyarrow.parquet as pq, json

def mb(p): return p.stat().st_size / 1e6
rows = []
for r in REG:
    for f in sorted((REF / r).glob("*.parquet")):
        m = pq.ParquetFile(f).metadata
        rows.append(dict(region=r, file=f.name.replace(f"{r}-", ""), rows=m.num_rows, cols=m.num_columns, MB=mb(f)))
inv = pd.DataFrame(rows)
print(inv.pivot(index="file", columns="region", values="rows")[REG].fillna(0).astype(int).to_string())
print()
print(inv.pivot(index="file", columns="region", values="MB")[REG].round(1).to_string())

region                             maricopa-az  northern-ca  eastern-ok  south-central-tx  eastern-wa
file                                                                                                 
census-acs-housing.parquet                1593          591        1192              6010         415
overture-buildings.parquet             2908224      1164724     2551694          11463801     1081346
overture-infrastructure.parquet           9482         3318        2726             13622        4131
overture-pois.parquet                   300046       116897      207369           1300334       76491
overture-rail.parquet                     3830         2006        5066             17895        3473
overture-roads-unfiltered.parquet      1170165       479596     1037555           4122701      383136
overture-roads.parquet                  494492       191836      520550           1937009      142649

region                             maricopa-az  northern-ca  eastern-ok  south-ce

**Takeaway:** The package holds about 19.2 million buildings, 2.0 million places and 7.2 million unfiltered road segments (3.3 million after the drivable-class filter, so the filter removes more than half of all segments). South-Central Texas alone holds 60% of the buildings (11.5 million) and 65% of the places. Everything is region-major with the same file names, which makes the build in notebook 02 a loop over regions.

## 1. Is the scored tract list the same in every source?
The sample submission is the authoritative scored list. It should match the tract polygons, the ACS housing table and the frozen boundary list, and the national strata table should contain every scored tract.

In [2]:
ss = {r: pd.read_csv(REF / r / f"{r}-sample-submission.csv", dtype={"GEOID": str}) for r in REG}
poly = {r: pd.read_parquet(tracts_parquet("census-tracts", r), columns=["GEOID"]) for r in REG}
acs = {r: pd.read_parquet(REF / r / f"{r}-census-acs-housing.parquet") for r in REG}
bnd = {r: pd.read_csv(BND / f"{r}-tract-geoids.csv", dtype={"GEOID": str}) for r in REG}
nat = pd.read_parquet(tracts_parquet("strata-tract-table"), columns=["GEOID"])
natset = set(nat.GEOID.astype(str))
out = []
for r in REG:
    s = set(ss[r].GEOID)
    out.append(dict(region=r, scored=len(s), dup_ids=ss[r].GEOID.duplicated().sum(),
                    polygons=len(poly[r]), poly_eq_scored=set(poly[r].GEOID) == s,
                    acs=len(acs[r]), acs_covers_scored=s <= set(acs[r].GEOID.astype(str)),
                    boundary_list=len(bnd[r]), boundary_minus_scored=len(set(bnd[r].GEOID) - s), scored_minus_boundary=len(s - set(bnd[r].GEOID)),
                    in_national_strata=len(s & natset)))
chk = pd.DataFrame(out).set_index("region"); print(chk.T.to_string())
allids = pd.concat([ss[r].assign(region=r) for r in REG])
print("\nGEOID unique across regions:", allids.GEOID.is_unique, "| total rows", len(allids), "| scored four regions:", len(allids[allids.region.isin(SCORED)]))
print("GEOID length:", allids.GEOID.str.len().value_counts().to_dict())

region                maricopa-az northern-ca eastern-ok south-central-tx eastern-wa
scored                       1593         591       1192             6003        415
dup_ids                         0           0          0                0          0
polygons                     1593         591       1192             6010        415
poly_eq_scored               True        True       True            False       True
acs                          1593         591       1192             6010        415
acs_covers_scored            True        True       True             True       True
boundary_list                1593         591       1192             6010        415
boundary_minus_scored           0           0          0                7          0
scored_minus_boundary           0           0          0                0          0
in_national_strata           1593         591       1192             6003        415

GEOID unique across regions: True | total rows 9794 | scored fou

 The only mismatch is South-Central Texas, whose polygons, ACS table and boundary list hold 6,010 tracts against 6,003 scored. Scored rows are 9,379 in the four named regions plus 415 in Eastern WA.

## Which tracts are in the boundary list but not scored?
The package says seven South-Central Texas water tracts were dropped because nothing could be scored there. Check what they are, and check the state codes that define region membership.

In [3]:
dropped = sorted(set(bnd["south-central-tx"].GEOID) - set(ss["south-central-tx"].GEOID))
print("dropped from scored list:", dropped)
sp = pd.read_parquet(tracts_parquet("census-tracts", "south-central-tx"))
print(sp[sp.GEOID.isin(dropped)][[c for c in sp.columns if c not in ("geometry",)]].head(8).to_string())
print()
allids["state"] = allids.GEOID.str[:2]
print(allids.groupby(["region", "state"]).size().rename("tracts").to_string())

dropped from scored list: ['48007990000', '48061990000', '48261990000', '48273990000', '48321990000', '48355990000', '48489990000']


            GEOID STATEFP COUNTYFP TRACTCE           NAMELSAD  ALAND  pop_total  pop_urban  pop_rural  pct_urban ur_class  frac_inside_aoi                                                                            bbox
4765  48321990000      48      321  990000  Census Tract 9900      0          0          0          0        NaN     None            1.000   {'xmin': -96.395198, 'ymin': 28.279736, 'xmax': -95.49565, 'ymax': 28.824493}
5521  48007990000      48      007  990000  Census Tract 9900      0          0          0          0        NaN     None            1.000   {'xmin': -97.04165, 'ymin': 27.803783, 'xmax': -96.815905, 'ymax': 28.057483}
5538  48355990000      48      355  990000  Census Tract 9900      0          0          0          0        NaN     None            1.000    {'xmin': -97.218072, 'ymin': 27.5683, 'xmax': -96.984281, 'ymax': 27.835447}
5548  48273990000      48      273  990000  Census Tract 9900      0          0          0          0        NaN     None   

## 3. What does the submission template look like?


In [4]:
for r in REG:
    d = ss[r]
    print(r, d.shape, list(d.columns), "| value ranges:", {c: (float(d[c].min()), float(d[c].max())) for c in d.columns if c != "GEOID"}, "| nulls:", int(d.isna().sum().sum()))
print()
print(ss["eastern-ok"].head(3).to_string(index=False))
print("distinct values in coverage_gap_score of the template:", pd.concat(ss.values()).coverage_gap_score.unique()[:5])

maricopa-az (1593, 5) ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score'] | value ranges: {'transport_gap': (0.0, 0.0), 'building_gap': (0.0, 0.0), 'poi_gap': (0.0, 0.0), 'coverage_gap_score': (0.0, 0.0)} | nulls: 0
northern-ca (591, 5) ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score'] | value ranges: {'transport_gap': (0.0, 0.0), 'building_gap': (0.0, 0.0), 'poi_gap': (0.0, 0.0), 'coverage_gap_score': (0.0, 0.0)} | nulls: 0
eastern-ok (1192, 5) ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score'] | value ranges: {'transport_gap': (0.0, 0.0), 'building_gap': (0.0, 0.0), 'poi_gap': (0.0, 0.0), 'coverage_gap_score': (0.0, 0.0)} | nulls: 0
south-central-tx (6003, 5) ['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score'] | value ranges: {'transport_gap': (0.0, 0.0), 'building_gap': (0.0, 0.0), 'poi_gap': (0.0, 0.0), 'coverage_gap_score': (0.0, 0.0)} | nulls: 0
eastern-wa (415, 5) ['GEOID', 'transp

## 4. The regions
Size and extent of the five study regions, from the area-of-interest polygons and the tract polygons.

In [5]:
import duckdb
c = duckdb.connect(); c.sql("load spatial"); c.sql("set geometry_always_xy = true")
reg_rows = []
for r in REG:
    q = f"""select count(*) n, sum(ALAND)/1e6 land_km2,
            min(ST_XMin(geometry)) x0, max(ST_XMax(geometry)) x1, min(ST_YMin(geometry)) y0, max(ST_YMax(geometry)) y1,
            min(frac_inside_aoi) min_frac_inside
            from read_parquet('{tracts_parquet("census-tracts", r).as_posix()}') where GEOID in (select GEOID from read_csv('{(REF / r / f"{r}-sample-submission.csv").as_posix()}', types={{'GEOID':'VARCHAR'}}))"""
    reg_rows.append(c.sql(q).df().assign(region=r))
R = pd.concat(reg_rows).set_index("region")
R["scored"] = [len(ss[r]) for r in REG]
print(R.rename(columns={"x0": "lon_min", "x1": "lon_max", "y0": "lat_min", "y1": "lat_max"}).to_string())
print("\nshare of the 9,379 scored tracts (four named regions):")
print((R.loc[SCORED, "n"] / R.loc[SCORED, "n"].sum()).round(3).to_string())

                     n    land_km2  lon_min  lon_max  lat_min  lat_max  min_frac_inside  scored
region                                                                                         
maricopa-az       1593 218,446.506 -114.564 -108.208   31.332   36.410            1.000    1593
northern-ca        591 110,680.881 -124.230 -119.996   38.313   42.010            1.000     591
eastern-ok        1192 160,555.056 -100.004  -94.431   33.616   37.002            1.000    1192
south-central-tx  6003 447,548.820 -103.253  -94.646   25.837   34.242            1.000    6003
eastern-wa         415 128,946.404 -122.284 -116.916   45.606   49.001            1.000     415

share of the 9,379 scored tracts (four named regions):
region
maricopa-az        0.170
northern-ca        0.063
eastern-ok         0.127
south-central-tx   0.640


## 5. What is in each Overture layer?


In [6]:
for layer in ["buildings", "roads", "roads-unfiltered", "rail", "infrastructure", "pois"]:
    f = REF / "eastern-ok" / f"eastern-ok-overture-{layer}.parquet"
    sch = pq.ParquetFile(f).schema_arrow
    print(f"{layer:17s} {len(sch.names):2d} cols:", ", ".join(sch.names))

buildings         26 cols: id, names, sources, level, height, min_height, is_underground, num_floors, num_floors_underground, min_floor, subtype, class, facade_color, facade_material, roof_material, roof_shape, roof_direction, roof_orientation, roof_color, roof_height, has_parts, version, theme, type, bbox, geometry
roads             23 cols: id, names, subtype, class, subclass, subclass_rules, connectors, road_surface, road_flags, rail_flags, width_rules, level_rules, access_restrictions, speed_limits, prohibited_transitions, routes, destinations, sources, version, theme, type, bbox, geometry
roads-unfiltered  23 cols: id, names, subtype, class, subclass, subclass_rules, connectors, road_surface, road_flags, rail_flags, width_rules, level_rules, access_restrictions, speed_limits, prohibited_transitions, routes, destinations, sources, version, theme, type, bbox, geometry
rail              23 cols: id, names, subtype, class, subclass, subclass_rules, connectors, road_surface, road_flags